In [12]:
import yfinance as yf
import pandas as pd

hourly_data = yf.download(tickers="AAPL", period="730d", interval="1h")


ticker = yf.Ticker("AAPL").history(period="730d", interval="1h")
ticker

[*********************100%***********************]  1 of 1 completed


,Open,High,Low,Close,Volume,Dividends,Stock Splits
Datetime,,,,,,,
2023-10-18 09:30:00-04:00,175.580002,177.574997,175.419998,176.619995,11694290,0.0,0.0
2023-10-18 10:30:00-04:00,176.604996,177.119995,175.755005,176.729996,7097323,0.0,0.0
2023-10-18 11:30:00-04:00,176.729996,177.529999,176.630096,176.820007,5348489,0.0,0.0
2023-10-18 12:30:00-04:00,176.820007,177.369995,176.225006,177.119995,5355115,0.0,0.0
2023-10-18 13:30:00-04:00,177.119995,177.130005,176.220001,176.695007,4533374,0.0,0.0
...,...,...,...,...,...,...,...
2026-09-15 12:30:00-04:00,330.400208,330.629211,329.420013,330.029999,2038493,0.0,0.0
2026-09-15 13:30:00-04:00,330.040009,330.829987,330.040009,330.410004,1734807,0.0,0.0
2026-09-15 14:30:00-04:00,330.399994,330.589996,329.829987,330.239990,2057605,0.0,0.0


In [15]:
# AAPL is the Apple ticker symbol (not APPL)
historical_data = yf.download(
    "AAPL",
    period="max",
    interval="1h",
    auto_adjust=False
)

data = pd.DataFrame(historical_data)
data

[*********************100%***********************]  1 of 1 completed


Price,Adj Close,Close,High,Low,Open,Volume
Ticker,AAPL,AAPL,AAPL,AAPL,AAPL,AAPL
Datetime,,,,,,
2024-09-16 09:30:00-04:00,216.660004,216.660004,217.199997,213.919998,216.470001,0
2024-09-16 10:30:00-04:00,216.556503,216.556503,216.914993,214.839996,216.640106,8162076
2024-09-16 11:30:00-04:00,215.399994,215.399994,216.580002,214.910004,216.550003,4852521
2024-09-16 12:30:00-04:00,215.964996,215.964996,215.979996,214.920105,215.389999,3748142
2024-09-16 13:30:00-04:00,216.429993,216.429993,216.520004,215.750000,215.960007,3606890
...,...,...,...,...,...,...
2026-09-15 12:30:00-04:00,330.029999,330.029999,330.629211,329.420013,330.400208,2038493
2026-09-15 13:30:00-04:00,330.410004,330.410004,330.829987,330.040009,330.040009,1734807


In [1]:
import sqlite3
import yfinance as yf
import pandas as pd
from datetime import datetime, timezone
import os

# ── Config ────────────────────────────────────────────────────────────────
DB_FILE = os.path.join(os.path.dirname(os.getcwd()), 'FredProject', 'macroscope.db')
# Adjust if needed — path relative to where you run the notebook:
DB_FILE = 'macroscope.db'

# ── DB helpers ────────────────────────────────────────────────────────────
def get_conn():
    conn = sqlite3.connect(DB_FILE)
    conn.execute('PRAGMA foreign_keys = ON')
    return conn

def migrate_db():
    """Add 'ticker' column to stock table if it doesn't exist yet."""
    conn = get_conn()
    cur = conn.cursor()
    # Check existing columns
    cols = [row[1] for row in cur.execute('PRAGMA table_info(stock)').fetchall()]
    if 'ticker' not in cols:
        cur.execute('ALTER TABLE stock ADD COLUMN ticker TEXT')
        print('✅ Added ticker column to stock table.')
    else:
        print('ℹ️  ticker column already exists — no migration needed.')
    conn.commit()
    conn.close()

migrate_db()
print(f'Database: {os.path.abspath(DB_FILE)}')


ℹ️  ticker column already exists — no migration needed.
Database: /Users/adamfilinger/FredProject/macroscope.db


In [2]:
# ── Watchlist helpers ────────────────────────────────────────────────────

def add_to_watchlist(ticker: str, name: str = None, currency: str = 'USD'):
    """
    Add a stock to the watchlist (stock table).
    If ticker already exists, does nothing.
    Name is auto-fetched from yfinance if not provided.
    """
    ticker = ticker.upper().strip()
    conn = get_conn()
    cur = conn.cursor()
    existing = cur.execute('SELECT ID FROM stock WHERE ticker = ?', (ticker,)).fetchone()
    if existing:
        print(f'ℹ️  {ticker} is already in the watchlist (ID={existing[0]}).')
        conn.close()
        return
    if name is None:
        try:
            info = yf.Ticker(ticker).info
            name = info.get('longName') or info.get('shortName') or ticker
            currency = info.get('currency', currency)
        except Exception:
            name = ticker
    cur.execute(
        'INSERT INTO stock (name, currency, ticker) VALUES (?, ?, ?)',
        (name, currency, ticker)
    )
    conn.commit()
    conn.close()
    print(f'✅ Added {ticker} ({name}) to watchlist.')

def remove_from_watchlist(ticker: str):
    """Remove a stock and all its price data from the DB."""
    ticker = ticker.upper().strip()
    conn = get_conn()
    cur = conn.cursor()
    row = cur.execute('SELECT ID FROM stock WHERE ticker = ?', (ticker,)).fetchone()
    if not row:
        print(f'⚠️  {ticker} not found in watchlist.')
        conn.close()
        return
    stock_id = row[0]
    cur.execute('DELETE FROM price_data WHERE stock_id = ?', (stock_id,))
    cur.execute('DELETE FROM stock WHERE ID = ?', (stock_id,))
    conn.commit()
    conn.close()
    print(f'🗑️  Removed {ticker} and all its price data.')

def get_watchlist() -> pd.DataFrame:
    """Return the current watchlist as a DataFrame."""
    conn = get_conn()
    df = pd.read_sql_query(
        'SELECT ID, ticker, name, currency, last_updated FROM stock ORDER BY ticker',
        conn
    )
    conn.close()
    return df

# ── Add your stocks here ──────────────────────────────────────────────────
add_to_watchlist('AAPL')   # Apple
add_to_watchlist('MSFT')   # Microsoft
add_to_watchlist('GOOGL')  # Alphabet
add_to_watchlist('NVDA')   # NVIDIA

print()
print('Current watchlist:')
get_watchlist()


✅ Added AAPL (Apple Inc.) to watchlist.
✅ Added MSFT (Microsoft Corporation) to watchlist.
✅ Added GOOGL (Alphabet Inc.) to watchlist.
✅ Added NVDA (NVIDIA Corporation) to watchlist.

Current watchlist:


,ID,ticker,name,currency,last_updated
0,1,AAPL,Apple Inc.,USD,None
1,3,GOOGL,Alphabet Inc.,USD,None
2,2,MSFT,Microsoft Corporation,USD,None
3,4,NVDA,NVIDIA Corporation,USD,None


In [3]:
# ── Download & store price data ───────────────────────────────────────────

INTERVAL = '1h'           # Hourly candles
FALLBACK_PERIOD = '730d'  # Used when no prior data exists for a ticker

def get_last_stored_time(stock_id: int) -> str | None:
    """Return the most recent stored timestamp for a given stock_id, or None."""
    conn = get_conn()
    row = conn.execute(
        'SELECT MAX(time) FROM price_data WHERE stock_id = ?', (stock_id,)
    ).fetchone()
    conn.close()
    return row[0] if row else None

def download_and_save(ticker: str, stock_id: int):
    """
    Download hourly price data for ticker via yfinance.
    Uses incremental strategy: only fetches rows newer than last stored timestamp.
    Saves Close price to price_data table.
    """
    last_time = get_last_stored_time(stock_id)

    if last_time:
        # yfinance 'start' param uses dates; parse last stored time
        start_dt = pd.Timestamp(last_time).tz_localize('UTC') if pd.Timestamp(last_time).tzinfo is None else pd.Timestamp(last_time)
        start_str = start_dt.strftime('%Y-%m-%d')
        print(f'  📥 {ticker}: incremental fetch from {start_str}')
        hist = yf.Ticker(ticker).history(start=start_str, interval=INTERVAL)
    else:
        print(f'  📥 {ticker}: full fetch ({FALLBACK_PERIOD})')
        hist = yf.Ticker(ticker).history(period=FALLBACK_PERIOD, interval=INTERVAL)

    if hist.empty:
        print(f'  ⚠️  {ticker}: no data returned.')
        return 0

    # Normalise index to UTC-aware ISO strings for consistent storage
    hist.index = pd.DatetimeIndex(hist.index).tz_convert('UTC')

    rows = []
    for ts, row in hist.iterrows():
        ts_str = ts.isoformat()
        # Skip rows already in DB (handles overlap when start_str == last date)
        if last_time and ts_str <= last_time:
            continue
        close_price = row.get('Close', None)
        if close_price is None or pd.isna(close_price):
            continue
        rows.append((stock_id, ts_str, int(round(close_price * 100))))
        # price stored as integer cents to avoid float precision issues

    if not rows:
        print(f'  ✅ {ticker}: already up-to-date.')
        return 0

    conn = get_conn()
    conn.executemany(
        'INSERT OR IGNORE INTO price_data (stock_id, time, price) VALUES (?, ?, ?)',
        rows
    )
    # Update last_updated on stock row
    conn.execute(
        'UPDATE stock SET last_updated = ? WHERE ID = ?',
        (datetime.now(timezone.utc).isoformat(), stock_id)
    )
    conn.commit()
    conn.close()
    print(f'  ✅ {ticker}: saved {len(rows)} new rows.')
    return len(rows)


def update_all():
    """Download and save price data for every stock in the watchlist."""
    watchlist = get_watchlist()
    if watchlist.empty:
        print('Watchlist is empty — add stocks first.')
        return
    print(f'Updating {len(watchlist)} stock(s)...\n')
    total = 0
    for _, s in watchlist.iterrows():
        total += download_and_save(s['ticker'], s['ID'])
    print(f'\nDone. {total} new rows inserted in total.')


# ── Run the update ────────────────────────────────────────────────────────
update_all()


Updating 4 stock(s)...

  📥 AAPL: full fetch (730d)
  ✅ AAPL: saved 5079 new rows.
  📥 GOOGL: full fetch (730d)
  ✅ GOOGL: saved 5079 new rows.
  📥 MSFT: full fetch (730d)
  ✅ MSFT: saved 5079 new rows.
  📥 NVDA: full fetch (730d)
  ✅ NVDA: saved 5079 new rows.

Done. 20316 new rows inserted in total.


In [4]:
# ── Inspect stored price data ────────────────────────────────────────────

def load_price_data(ticker: str) -> pd.DataFrame:
    """
    Load stored price data for a ticker from the DB.
    Returns a DataFrame with a 'datetime' index and 'close' column (in real price).
    """
    ticker = ticker.upper().strip()
    conn = get_conn()
    df = pd.read_sql_query(
        '''
        SELECT pd.time, pd.price
        FROM price_data pd
        JOIN stock s ON s.ID = pd.stock_id
        WHERE s.ticker = ?
        ORDER BY pd.time ASC
        ''',
        conn,
        params=(ticker,)
    )
    conn.close()
    if df.empty:
        print(f'No data found for {ticker}.')
        return df
    df['time'] = pd.to_datetime(df['time'], utc=True)
    df = df.set_index('time')
    df['close'] = df['price'] / 100.0  # convert cents back to dollars
    df = df.drop(columns=['price'])
    return df

# Summary table: rows per ticker
conn = get_conn()
summary = pd.read_sql_query(
    '''
    SELECT s.ticker, s.name, COUNT(pd.time) AS rows_stored, s.last_updated
    FROM stock s
    LEFT JOIN price_data pd ON pd.stock_id = s.ID
    GROUP BY s.ID
    ORDER BY s.ticker
    ''',
    conn
)
conn.close()
print('Storage summary:')
display(summary)

# Preview AAPL data
aapl = load_price_data('AAPL')
print(f'\nAAPL — {len(aapl)} hourly rows stored')
aapl


Storage summary:


,ticker,name,rows_stored,last_updated
0,AAPL,Apple Inc.,5079,2026-09-25T16:49:05.722789+00:00
1,GOOGL,Alphabet Inc.,5079,2026-09-25T16:49:06.300066+00:00
2,MSFT,Microsoft Corporation,5079,2026-09-25T16:49:07.490867+00:00
3,NVDA,NVIDIA Corporation,5079,2026-09-25T16:49:08.550953+00:00



AAPL — 5079 hourly rows stored


,close
time,
2023-10-27 13:30:00+00:00,168.16
2023-10-27 14:30:00+00:00,168.74
2023-10-27 15:30:00+00:00,167.76
2023-10-27 16:30:00+00:00,167.19
2023-10-27 17:30:00+00:00,167.70
...,...
2026-09-24 19:30:00+00:00,335.87
2026-09-25 13:30:00+00:00,336.36
2026-09-25 14:30:00+00:00,338.42


In [1]:
# ── Signal Analysis: BUY / HOLD / SELL ──────────────────────────────────
# Algorithm: EMA Crossover (20 vs 50 periods) filtered by RSI(14)
#
#  BUY  — EMA20 > EMA50  AND  40 < RSI < 70  (uptrend, not overbought)
#  SELL — EMA20 < EMA50  AND  RSI < 45       (downtrend, momentum confirms)
#  HOLD — everything else
#
# All calculations run on the hourly Close prices stored in price_data.

import sqlite3
import pandas as pd
import numpy as np

# Re-use DB_FILE from the setup cell; define here as fallback
try:
    DB_FILE  # noqa: F821
except NameError:
    DB_FILE = 'macroscope.db'


# ── Indicator helpers ────────────────────────────────────────────────────

def compute_ema(series: pd.Series, span: int) -> pd.Series:
    """Exponential Moving Average."""
    return series.ewm(span=span, adjust=False).mean()


def compute_rsi(series: pd.Series, period: int = 14) -> pd.Series:
    """
    Wilder's RSI.  Returns values in [0, 100].
    Uses Exponential smoothing (equivalent to Wilder's method via alpha=1/period).
    """
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = (-delta).clip(lower=0)
    avg_gain = gain.ewm(alpha=1 / period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1 / period, adjust=False).mean()
    rs = avg_gain / avg_loss.replace(0, np.nan)
    rsi = 100 - (100 / (1 + rs))
    return rsi


def generate_signal(close: pd.Series,
                    ema_fast: int = 20,
                    ema_slow: int = 50,
                    rsi_period: int = 14) -> dict:
    """
    Compute EMA crossover + RSI signal on a Close price series.

    Returns a dict with:
      signal       – 'BUY' | 'HOLD' | 'SELL'
      last_close   – most recent closing price
      ema_fast     – latest fast EMA value
      ema_slow     – latest slow EMA value
      rsi          – latest RSI value
      trend        – 'UP' | 'DOWN'
    """
    if len(close) < ema_slow + rsi_period:
        return {
            'signal': 'INSUFFICIENT DATA',
            'last_close': None, 'ema_fast': None,
            'ema_slow': None, 'rsi': None, 'trend': None,
        }

    fast = compute_ema(close, ema_fast)
    slow = compute_ema(close, ema_slow)
    rsi  = compute_rsi(close, rsi_period)

    last_close = close.iloc[-1]
    last_fast  = fast.iloc[-1]
    last_slow  = slow.iloc[-1]
    last_rsi   = rsi.iloc[-1]
    trend      = 'UP' if last_fast > last_slow else 'DOWN'

    if trend == 'UP' and 40 < last_rsi < 70:
        signal = 'BUY'
    elif trend == 'DOWN' and last_rsi < 45:
        signal = 'SELL'
    else:
        signal = 'HOLD'

    return {
        'signal':     signal,
        'last_close': round(last_close, 2),
        'ema_fast':   round(last_fast,  2),
        'ema_slow':   round(last_slow,  2),
        'rsi':        round(last_rsi,   1),
        'trend':      trend,
    }


# ── Load each watchlist stock and evaluate ────────────────────────────────

def analyse_watchlist() -> pd.DataFrame:
    conn = sqlite3.connect(DB_FILE)
    watchlist = pd.read_sql_query(
        'SELECT ID, ticker, name FROM stock ORDER BY ticker', conn
    )

    records = []
    for _, row in watchlist.iterrows():
        price_df = pd.read_sql_query(
            'SELECT time, price FROM price_data WHERE stock_id = ? ORDER BY time ASC',
            conn, params=(row['ID'],)
        )
        if price_df.empty:
            records.append({'ticker': row['ticker'], 'name': row['name'],
                            'signal': 'NO DATA', 'last_close': None,
                            'ema_fast': None, 'ema_slow': None,
                            'rsi': None, 'trend': None})
            continue

        close = price_df['price'] / 100.0  # cents -> dollars
        result = generate_signal(close)
        records.append({'ticker': row['ticker'], 'name': row['name'], **result})

    conn.close()
    return pd.DataFrame(records)


results = analyse_watchlist()

# ── Pretty display ────────────────────────────────────────────────────────

EMOJI = {'BUY': '🟢 BUY', 'SELL': '🔴 SELL', 'HOLD': '🟡 HOLD',
         'NO DATA': '⚪ NO DATA', 'INSUFFICIENT DATA': '⚪ INSUFFICIENT DATA'}

display_df = results.copy()
display_df['signal'] = display_df['signal'].map(lambda s: EMOJI.get(s, s))
display_df = display_df.rename(columns={
    'ticker': 'Ticker', 'name': 'Name', 'signal': 'Signal',
    'last_close': 'Last Close ($)', 'ema_fast': 'EMA20',
    'ema_slow': 'EMA50', 'rsi': 'RSI(14)', 'trend': 'Trend',
})

print('=== Watchlist Signal Report ===')
print(f'Generated at: {pd.Timestamp.now().strftime("%Y-%m-%d %H:%M")}\n')
display(display_df.set_index('Ticker'))


=== Watchlist Signal Report ===
Generated at: 2026-09-25 18:55



,Name,Signal,Last Close ($),EMA20,EMA50,RSI(14),Trend
Ticker,,,,,,,
AAPL,Apple Inc.,🟢 BUY,338.91,337.83,336.28,54.8,UP
GOOGL,Alphabet Inc.,🟡 HOLD,343.41,343.60,345.05,47.5,DOWN
MSFT,Microsoft Corporation,🟡 HOLD,517.12,503.22,499.68,77.5,UP
NVDA,NVIDIA Corporation,🟢 BUY,224.85,224.49,223.33,52.6,UP


In [2]:
import sqlite3, os

try:
    DB_FILE  # noqa: F821
except NameError:
    DB_FILE = "macroscope.db"

CREATE_SQL = """
CREATE TABLE IF NOT EXISTS stock_financials (
    id                          INTEGER PRIMARY KEY AUTOINCREMENT,
    stock_id                    INTEGER NOT NULL,
    snapshot_date               TEXT NOT NULL,

    -- Valuation
    market_cap                  REAL,
    enterprise_value            REAL,
    trailing_pe                 REAL,
    forward_pe                  REAL,
    peg_ratio                   REAL,
    price_to_book               REAL,
    price_to_sales_ttm          REAL,
    ev_to_revenue               REAL,
    ev_to_ebitda                REAL,

    -- Profitability
    profit_margins              REAL,
    gross_margins               REAL,
    ebitda_margins              REAL,
    operating_margins           REAL,
    return_on_assets            REAL,
    return_on_equity            REAL,

    -- Growth
    earnings_growth             REAL,
    revenue_growth              REAL,
    earnings_quarterly_growth   REAL,

    -- Income / Cash
    total_revenue               REAL,
    gross_profits               REAL,
    ebitda                      REAL,
    net_income                  REAL,
    free_cashflow               REAL,
    operating_cashflow          REAL,

    -- Balance Sheet
    total_cash                  REAL,
    total_cash_per_share        REAL,
    total_debt                  REAL,
    debt_to_equity              REAL,
    quick_ratio                 REAL,
    current_ratio               REAL,

    -- Per-Share
    trailing_eps                REAL,
    forward_eps                 REAL,
    book_value                  REAL,
    revenue_per_share           REAL,

    -- Dividends
    dividend_rate               REAL,
    dividend_yield              REAL,
    payout_ratio                REAL,
    five_year_avg_div_yield     REAL,

    -- Risk / Market
    beta                        REAL,
    fifty_two_week_high         REAL,
    fifty_two_week_low          REAL,
    shares_outstanding          REAL,
    float_shares                REAL,
    shares_short                REAL,
    short_ratio                 REAL,
    short_percent_of_float      REAL,

    -- Analyst Targets
    target_high_price           REAL,
    target_low_price            REAL,
    target_mean_price           REAL,
    target_median_price         REAL,
    recommendation_key          TEXT,
    recommendation_mean         REAL,
    num_analyst_opinions        INTEGER,

    FOREIGN KEY (stock_id) REFERENCES stock(ID),
    UNIQUE (stock_id, snapshot_date)
)
"""

def migrate_financials_table():
    conn = sqlite3.connect(DB_FILE)
    conn.execute("PRAGMA foreign_keys = ON")
    conn.execute(CREATE_SQL)
    conn.commit()
    conn.close()
    print("stock_financials table ready.")

migrate_financials_table()


stock_financials table ready.


In [3]:
import yfinance as yf
import sqlite3
import pandas as pd
from datetime import date

try:
    DB_FILE  # noqa: F821
except NameError:
    DB_FILE = "macroscope.db"

# Map DB column -> yfinance .info key
INFO_FIELD_MAP = {
    # Valuation
    "market_cap":                "marketCap",
    "enterprise_value":          "enterpriseValue",
    "trailing_pe":               "trailingPE",
    "forward_pe":                "forwardPE",
    "peg_ratio":                 "pegRatio",
    "price_to_book":             "priceToBook",
    "price_to_sales_ttm":        "priceToSalesTrailing12Months",
    "ev_to_revenue":             "enterpriseToRevenue",
    "ev_to_ebitda":              "enterpriseToEbitda",
    # Profitability
    "profit_margins":            "profitMargins",
    "gross_margins":             "grossMargins",
    "ebitda_margins":            "ebitdaMargins",
    "operating_margins":         "operatingMargins",
    "return_on_assets":          "returnOnAssets",
    "return_on_equity":          "returnOnEquity",
    # Growth
    "earnings_growth":           "earningsGrowth",
    "revenue_growth":            "revenueGrowth",
    "earnings_quarterly_growth": "earningsQuarterlyGrowth",
    # Income / Cash
    "total_revenue":             "totalRevenue",
    "gross_profits":             "grossProfits",
    "ebitda":                    "ebitda",
    "net_income":                "netIncomeToCommon",
    "free_cashflow":             "freeCashflow",
    "operating_cashflow":        "operatingCashflow",
    # Balance Sheet
    "total_cash":                "totalCash",
    "total_cash_per_share":      "totalCashPerShare",
    "total_debt":                "totalDebt",
    "debt_to_equity":            "debtToEquity",
    "quick_ratio":               "quickRatio",
    "current_ratio":             "currentRatio",
    # Per-Share
    "trailing_eps":              "trailingEps",
    "forward_eps":               "forwardEps",
    "book_value":                "bookValue",
    "revenue_per_share":         "revenuePerShare",
    # Dividends
    "dividend_rate":             "dividendRate",
    "dividend_yield":            "dividendYield",
    "payout_ratio":              "payoutRatio",
    "five_year_avg_div_yield":   "fiveYearAvgDividendYield",
    # Risk / Market
    "beta":                      "beta",
    "fifty_two_week_high":       "fiftyTwoWeekHigh",
    "fifty_two_week_low":        "fiftyTwoWeekLow",
    "shares_outstanding":        "sharesOutstanding",
    "float_shares":              "floatShares",
    "shares_short":              "sharesShort",
    "short_ratio":               "shortRatio",
    "short_percent_of_float":    "shortPercentOfFloat",
    # Analyst Targets
    "target_high_price":         "targetHighPrice",
    "target_low_price":          "targetLowPrice",
    "target_mean_price":         "targetMeanPrice",
    "target_median_price":       "targetMedianPrice",
    "recommendation_key":        "recommendationKey",
    "recommendation_mean":       "recommendationMean",
    "num_analyst_opinions":      "numberOfAnalystOpinions",
}

DB_COLS = list(INFO_FIELD_MAP.keys())


def fetch_financials(ticker: str) -> dict | None:
    """Download .info from yfinance and extract the mapped fields."""
    try:
        info = yf.Ticker(ticker).info
    except Exception as e:
        print(f"  {ticker}: could not fetch info -- {e}")
        return None
    if not info or info.get("quoteType") is None:
        print(f"  {ticker}: empty or invalid info response.")
        return None
    row = {}
    for db_col, info_key in INFO_FIELD_MAP.items():
        val = info.get(info_key)
        if val is not None and not isinstance(val, str):
            try:
                val = float(val)
            except (TypeError, ValueError):
                val = None
        row[db_col] = val
    return row


def save_financials(stock_id: int, ticker: str, today: str) -> bool:
    """
    Fetch and store one financial snapshot for today.
    Incremental: skips if a row for (stock_id, today) already exists.
    """
    conn = sqlite3.connect(DB_FILE)
    conn.execute("PRAGMA foreign_keys = ON")
    existing = conn.execute(
        "SELECT id FROM stock_financials WHERE stock_id = ? AND snapshot_date = ?",
        (stock_id, today)
    ).fetchone()
    if existing:
        print(f"  {ticker}: snapshot for {today} already exists -- skipping.")
        conn.close()
        return False

    row = fetch_financials(ticker)
    if row is None:
        conn.close()
        return False

    cols         = ["stock_id", "snapshot_date"] + DB_COLS
    values       = [stock_id, today] + [row[c] for c in DB_COLS]
    placeholders = ", ".join(["?"] * len(cols))
    col_names    = ", ".join(cols)

    conn.execute(
        f"INSERT OR IGNORE INTO stock_financials ({col_names}) VALUES ({placeholders})",
        values
    )
    conn.commit()
    conn.close()

    non_null = sum(1 for v in row.values() if v is not None)
    print(f"  {ticker}: saved snapshot for {today} ({non_null}/{len(DB_COLS)} fields populated).")
    return True


def update_all_financials():
    """Download and store financial info for every stock in the watchlist."""
    today = date.today().isoformat()
    conn  = sqlite3.connect(DB_FILE)
    watchlist = pd.read_sql_query("SELECT ID, ticker FROM stock ORDER BY ticker", conn)
    conn.close()

    if watchlist.empty:
        print("Watchlist is empty -- add stocks first.")
        return

    print(f"Fetching financials for {len(watchlist)} stock(s) -- snapshot date: {today}\n")
    saved = 0
    for _, s in watchlist.iterrows():
        if save_financials(s["ID"], s["ticker"], today):
            saved += 1
    print(f"\nDone. {saved} new snapshot(s) saved.")


# Run
update_all_financials()


Fetching financials for 4 stock(s) -- snapshot date: 2026-09-25

  AAPL: saved snapshot for 2026-09-25 (53/53 fields populated).
  GOOGL: saved snapshot for 2026-09-25 (52/53 fields populated).
  MSFT: saved snapshot for 2026-09-25 (53/53 fields populated).
  NVDA: saved snapshot for 2026-09-25 (53/53 fields populated).

Done. 4 new snapshot(s) saved.


In [4]:
import sqlite3
import pandas as pd

try:
    DB_FILE  # noqa: F821
except NameError:
    DB_FILE = "macroscope.db"


def load_latest_financials() -> pd.DataFrame:
    """
    Returns a DataFrame with the most recent snapshot per stock.
    Pivoted so rows = metrics, columns = tickers (easy side-by-side comparison).
    """
    conn = sqlite3.connect(DB_FILE)
    df = pd.read_sql_query(
        """
        SELECT s.ticker, f.*
        FROM stock_financials f
        JOIN stock s ON s.ID = f.stock_id
        WHERE f.snapshot_date = (
            SELECT MAX(f2.snapshot_date)
            FROM stock_financials f2
            WHERE f2.stock_id = f.stock_id
        )
        ORDER BY s.ticker
        """,
        conn
    )
    conn.close()
    return df


def load_financials_history(ticker: str) -> pd.DataFrame:
    """Return all snapshots for a single ticker, ordered by date."""
    ticker = ticker.upper().strip()
    conn = sqlite3.connect(DB_FILE)
    df = pd.read_sql_query(
        """
        SELECT f.*
        FROM stock_financials f
        JOIN stock s ON s.ID = f.stock_id
        WHERE s.ticker = ?
        ORDER BY f.snapshot_date ASC
        """,
        conn,
        params=(ticker,)
    )
    conn.close()
    return df.set_index("snapshot_date") if not df.empty else df


# Side-by-side comparison of the latest snapshot
latest = load_latest_financials()

if latest.empty:
    print("No financial snapshots found. Run update_all_financials() first.")
else:
    drop_cols = ["id", "stock_id", "snapshot_date"]
    pivot = (
        latest
        .drop(columns=[c for c in drop_cols if c in latest.columns])
        .set_index("ticker")
        .T
    )

    GROUPS = {
        "Valuation":     ["market_cap", "enterprise_value", "trailing_pe", "forward_pe",
                          "peg_ratio", "price_to_book", "price_to_sales_ttm",
                          "ev_to_revenue", "ev_to_ebitda"],
        "Profitability": ["profit_margins", "gross_margins", "ebitda_margins",
                          "operating_margins", "return_on_assets", "return_on_equity"],
        "Growth":        ["earnings_growth", "revenue_growth", "earnings_quarterly_growth"],
        "Income/Cash":   ["total_revenue", "gross_profits", "ebitda", "net_income",
                          "free_cashflow", "operating_cashflow"],
        "Balance Sheet": ["total_cash", "total_debt", "debt_to_equity",
                          "quick_ratio", "current_ratio"],
        "Per Share":     ["trailing_eps", "forward_eps", "book_value", "revenue_per_share"],
        "Dividends":     ["dividend_rate", "dividend_yield", "payout_ratio",
                          "five_year_avg_div_yield"],
        "Risk/Short":    ["beta", "fifty_two_week_high", "fifty_two_week_low",
                          "short_ratio", "short_percent_of_float"],
        "Analyst":       ["target_high_price", "target_low_price", "target_mean_price",
                          "recommendation_key", "recommendation_mean", "num_analyst_opinions"],
    }

    snap_date = latest["snapshot_date"].iloc[0]
    print("=== Latest Financial Snapshot", snap_date, "===")
    print()
    for group, fields in GROUPS.items():
        in_pivot = [f for f in fields if f in pivot.index]
        if not in_pivot:
            continue
        print("--", group, "--")
        display(pivot.loc[in_pivot])
        print()


=== Latest Financial Snapshot 2026-09-25 ===

-- Valuation --


ticker,AAPL,GOOGL,MSFT,NVDA
market_cap,4961801863168.0,4212111966208.0,3832769413120.0,5434282344448.0
enterprise_value,4924421701632.0,4083402670080.0,3749371969536.0,5388768903168.0
trailing_pe,38.944447,17.280983,28.723427,28.451328
forward_pe,35.46923,23.115992,21.801088,14.350272
peg_ratio,2.7,1.22,1.62,0.48
price_to_book,46.19361,6.766937,8.665491,23.731943
price_to_sales_ttm,10.628872,9.447859,11.550087,17.9367
ev_to_revenue,10.549,9.158,11.299,17.786
ev_to_ebitda,29.319,23.581,19.303,26.774



-- Profitability --


ticker,AAPL,GOOGL,MSFT,NVDA
profit_margins,0.27619,0.54771,0.40305,0.63663
gross_margins,0.48653,0.60897,0.67944,0.74674
ebitda_margins,0.35979,0.38838,0.58534,0.66431
operating_margins,0.32623,0.34033,0.45111,0.66237
return_on_assets,0.27082,0.12959,0.14088,0.53572
return_on_equity,1.48751,0.48676,0.34039,1.17211



-- Growth --


...

ticker,AAPL,GOOGL,MSFT,NVDA
dividend_rate,1.08,0.88,3.92,1.0
dividend_yield,0.32,0.26,0.79,0.45
payout_ratio,0.1204,0.0426,0.1983,0.0354
five_year_avg_div_yield,0.5,NaN,0.79,0.05



-- Risk/Short --


ticker,AAPL,GOOGL,MSFT,NVDA
beta,1.085,1.225,1.108,2.217
fifty_two_week_high,345.34,408.61,553.72,236.54
fifty_two_week_low,243.42,235.84,349.2,164.27
short_ratio,3.03,3.68,3.21,2.29
short_percent_of_float,0.0088,0.0149,0.0091,0.0127



-- Analyst --


ticker,AAPL,GOOGL,MSFT,NVDA
target_high_price,405.0,515.0,870.0,515.0
target_low_price,215.0,340.0,440.0,180.0
target_mean_price,328.22205,429.45557,577.26135,327.7
recommendation_key,buy,strong_buy,strong_buy,strong_buy
recommendation_mean,2.20455,1.37705,1.32727,1.29508
num_analyst_opinions,39,54,52,59
